# Product Price Copilot (v2) - OpenAI Fine-tuning

This notebook builds a minimal **Product Price Copilot v2** that predicts:

- `price_min`
- `price_max`
- `confidence` (0 to 1)
- `rationale_tags` (fixed tags only, no free-text rationale)

It reuses your week6 dataset:

- `week6/human_in.csv`
- `week6/human_out.csv`

Then it creates fine-tuning JSONL files and includes OpenAI job + inference scaffolding.

In [22]:
import os
import re
import json
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / 'human_in.csv').exists():
    BASE_DIR = BASE_DIR / 'week6'

IN_PATH = BASE_DIR / 'human_in.csv'
OUT_PATH = BASE_DIR / 'human_out.csv'
JSONL_DIR = BASE_DIR / 'jsonl'
BASE_DIR

if not (IN_PATH.exists() and OUT_PATH.exists()):
    raise FileNotFoundError('Could not find human_in.csv and human_out.csv from current path.')

JSONL_DIR.mkdir(parents=True, exist_ok=True)

print('Input path:', IN_PATH)
print('Output path:', OUT_PATH)
print('JSONL output dir:', JSONL_DIR)

Input path: /Users/user/Desktop/apps/learn/andela-ai/llm_engineering/week6/human_in.csv
Output path: /Users/user/Desktop/apps/learn/andela-ai/llm_engineering/week6/human_out.csv
JSONL output dir: /Users/user/Desktop/apps/learn/andela-ai/llm_engineering/week6/jsonl


In [ ]:
# human_in.csv has a product text blob column + a placeholder column
human_in = pd.read_csv(IN_PATH, header=None, names=['product_blob', 'unused'])
# human_out.csv has the same product blob + numeric price
human_out = pd.read_csv(OUT_PATH, header=None, names=['product_blob', 'price'])

# Keep only rows with valid numeric prices
human_out['price'] = pd.to_numeric(human_out['price'], errors='coerce')
human_out = human_out.dropna(subset=['price']).reset_index(drop=True)

# Merge on product text to align features and label
df = human_in[['product_blob']].merge(
    human_out[['product_blob', 'price']],
    on='product_blob',
    how='inner'
)

print('Rows ready for v2:', len(df))
df.head(3)

In [ ]:
def parse_product_blob(blob: str) -> dict:
    fields = {
        'title': '',
        'category': '',
        'brand': '',
        'description': '',
        'details': ''
    }
    for line in str(blob).splitlines():
        if ':' not in line:
            continue
        key, value = line.split(':', 1)
        key = key.strip().lower()
        if key in fields:
            fields[key] = value.strip()
    return fields


def infer_rationale_tags(fields: dict, price: float) -> list:
    tags = []
    category = fields.get('category', '').lower()
    details = (fields.get('details', '') + ' ' + fields.get('description', '')).lower()
    brand = fields.get('brand', '').strip().lower()

    premium_brands = {
        'bose', 'sony', 'apple', 'samsung', 'canon', 'nikon', 'dyson',
        'rockford fosgate', 'thermaltake'
    }

    if any(x in category for x in ['electronics', 'computer', 'audio', 'headphone']):
        tags.append('electronics')
    if any(x in category for x in ['automotive', 'car', 'motorcycle']):
        tags.append('automotive')
    if any(x in category for x in ['home', 'hardware', 'kitchen']):
        tags.append('home_goods')
    if any(x in category for x in ['accessories', 'accessory']):
        tags.append('accessory')
    if any(x in category for x in ['collectible', 'art']):
        tags.append('niche_collectible')

    if brand in premium_brands:
        tags.append('premium_brand')
    elif not brand or brand in {'generic', 'unknown'}:
        tags.append('unknown_brand')

    if any(x in details for x in ['warranty', 'certified', 'wireless', 'bluetooth', 'professional', 'pro']):
        tags.append('high_spec_features')

    if price < 25:
        tags.append('budget_item')
    elif price > 250:
        tags.append('premium_priced')

    if not tags:
        tags.append('general_merch')

    return sorted(set(tags))


def build_target(price: float, tags: list) -> dict:
    if price < 20:
        pct = 0.30
    elif price < 100:
        pct = 0.20
    elif price < 300:
        pct = 0.15
    else:
        pct = 0.12

    price_min = round(max(1.0, price * (1 - pct)), 2)
    price_max = round(price * (1 + pct), 2)

    base_conf = {0.30: 0.62, 0.20: 0.72, 0.15: 0.80, 0.12: 0.86}[pct]
    if 'premium_brand' in tags:
        base_conf += 0.03
    if 'unknown_brand' in tags:
        base_conf -= 0.05

    confidence = round(min(0.95, max(0.50, base_conf)), 2)

    return {
        'price_min': price_min,
        'price_max': price_max,
        'confidence': confidence,
        'rationale_tags': tags
    }


def make_user_prompt(fields: dict) -> str:
    return (
        'Estimate this product price as a range and confidence. Return JSON only.\\n\\n'
        f"Title: {fields.get('title', '')}\\n"
        f"Category: {fields.get('category', '')}\\n"
        f"Brand: {fields.get('brand', '')}\\n"
        f"Description: {fields.get('description', '')}\\n"
        f"Details: {fields.get('details', '')}"
    )

In [ ]:
SYSTEM_PROMPT = (
    'You are Product Price Copilot v2. '
    'Always return strict JSON with keys: price_min, price_max, confidence, rationale_tags. '
    'rationale_tags must be from this set only: '
    '[electronics, automotive, home_goods, accessory, niche_collectible, premium_brand, '
    'unknown_brand, high_spec_features, budget_item, premium_priced, general_merch]. '
    'Do not return any free-text explanation.'
)

records = []
for row in df.itertuples(index=False):
    fields = parse_product_blob(row.product_blob)
    tags = infer_rationale_tags(fields, float(row.price))
    target = build_target(float(row.price), tags)

    record = {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': make_user_prompt(fields)},
            {'role': 'assistant', 'content': json.dumps(target, ensure_ascii=True)}
        ]
    }
    records.append(record)

print('Structured records created:', len(records))
print('Sample target:', records[0]['messages'][-1]['content'])

In [ ]:
# 80/20 split for minimal effort
seed = 42
rng = pd.Series(range(len(records))).sample(frac=1.0, random_state=seed).tolist()
shuffled = [records[i] for i in rng]

split_idx = int(len(shuffled) * 0.8)
train_records = shuffled[:split_idx]
val_records = shuffled[split_idx:]

train_path = JSONL_DIR / 'price_copilot_v2_train.jsonl'
val_path = JSONL_DIR / 'price_copilot_v2_validation.jsonl'


def write_jsonl(path: Path, rows: list) -> None:
    with path.open('w', encoding='utf-8') as f:
        for item in rows:
            f.write(json.dumps(item, ensure_ascii=True) + '\n')


write_jsonl(train_path, train_records)
write_jsonl(val_path, val_records)

print('Train size:', len(train_records), '| path:', train_path)
print('Validation size:', len(val_records), '| path:', val_path)

In [ ]:
# Quick sanity check: preview first two training lines
preview = []
with train_path.open('r', encoding='utf-8') as f:
    for _ in range(2):
        preview.append(next(f).strip())

for i, row in enumerate(preview, 1):
    print(f'Line {i}:')
    print(row[:500] + ('...' if len(row) > 500 else ''))
    print()

In [ ]:
# Fine-tune job scaffolding (run when OPENAI_API_KEY is set)
# pip install openai

from openai import OpenAI

client = OpenAI()

# Upload datasets
train_file = client.files.create(file=open(train_path, 'rb'), purpose='fine-tune')
val_file = client.files.create(file=open(val_path, 'rb'), purpose='fine-tune')

print('Train file id:', train_file.id)
print('Val file id:', val_file.id)

# Create fine-tuning job
BASE_MODEL = 'gpt-4.1-nano-2025-04-14'

job = client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model=BASE_MODEL,
    suffix='product-price-copilot-v2'
)

print('Fine-tune job id:', job.id)
print('Status:', job.status)

# Optional: track progress
# client.fine_tuning.jobs.retrieve(job.id)
# client.fine_tuning.jobs.list_events(fine_tuning_job_id=job.id, limit=20)

In [ ]:
# Inference helper for your fine-tuned model
# Replace with your returned fine-tuned model id

import json
from openai import OpenAI

client = OpenAI()
FT_MODEL = client.fine_tuning.jobs.retrieve(job.id).fine_tuned_model

INFER_SYSTEM = SYSTEM_PROMPT


def predict_price_v2(product_blob: str, model: str = FT_MODEL) -> dict:
    fields = parse_product_blob(product_blob)
    user_prompt = make_user_prompt(fields)

    resp = client.chat.completions.create(
        model=model,
        response_format={'type': 'json_object'},
        messages=[
            {'role': 'system', 'content': INFER_SYSTEM},
            {'role': 'user', 'content': user_prompt}
        ],
        temperature=0
    )

    raw = resp.choices[0].message.content
    return json.loads(raw)


# Example (uncomment after FT model is ready)
sample_blob = df.iloc[0]['product_blob']
pred = predict_price_v2(sample_blob)
pred

In [ ]:
# Minimal offline evaluation utility
# Use this after collecting predictions from the fine-tuned model.


def evaluate_range_predictions(y_true_prices, y_pred_structured):
    mids = []
    covered = 0

    for true_price, pred in zip(y_true_prices, y_pred_structured):
        lo = float(pred['price_min'])
        hi = float(pred['price_max'])
        mid = (lo + hi) / 2
        mids.append(mid)

        if lo <= float(true_price) <= hi:
            covered += 1

    mae_mid = sum(abs(t - p) for t, p in zip(y_true_prices, mids)) / len(y_true_prices)
    coverage = covered / len(y_true_prices)

    return {
        'mae_using_midpoint': round(mae_mid, 2),
        'range_coverage': round(coverage, 3)
    }


# Example usage after inference loop:
# Build validation rows aligned with the same shuffle/split used earlier.
df_shuffled = df.iloc[rng].reset_index(drop=True)
val_df = df_shuffled.iloc[split_idx:].reset_index(drop=True)

# Generate predictions first so `preds` exists.
# Keep this small to reduce API cost.
MAX_EVAL_ROWS = min(10, len(val_df))
preds = [
    predict_price_v2(blob)
    for blob in val_df['product_blob'].head(MAX_EVAL_ROWS).tolist()
]

y_true = val_df['price'].head(MAX_EVAL_ROWS).astype(float).tolist()
metrics = evaluate_range_predictions(y_true, preds)
print(metrics)